In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, accuracy_score

In [2]:
# 1. DATA INGESTION & ROBUST PREPROCESSING
print("Step 1: Ingesting customer account ledger...")
df = pd.read_csv('/content/customer_churn_ann_dataset1_55d8b8a8-aa58-4c01-b459-d30aeda770d2_268917_.csv')

Step 1: Ingesting customer account ledger...


In [3]:
# Drop CustomerID as it is metadata and holds no behavioral value
df = df.drop(columns=['CustomerID'])

In [4]:
# DATA CLEANING: Fill missing values in InternetService with a fallback category
df['InternetService']=df['InternetService'].fillna('Unknown')

In [19]:
# Isolate features and target label
X_raw = df.drop(columns=['Churn', 'CustomerID'])
y = df['Churn'].map({'No': 0, 'Yes': 1}) # Encode binary target to integers

In [20]:
# Transform categorical strings into numerical structural flags via One-Hot Encoding
X_encoded = pd.get_dummies(X_raw, columns=['Contract', 'InternetService','TechSupport', 'PaymentMethod', 'PaperlessBilling'], drop_first=True)

# Ensure CustomerID is dropped if it passed through one-hot encoding untouched
if 'CustomerID' in X_encoded.columns:
    X_encoded = X_encoded.drop(columns=['CustomerID'])

In [21]:
# Save the exact schema template to align incoming manual traffic during live inference
TRAINING_FEATURE_SCHEMA = X_encoded.columns.tolist()

In [28]:
#Split data into 80% training / 20% evaluation sets
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_encoded,y,test_size=0.2,random_state=42,stratify=y
)

In [29]:
#2. Feature Scaling (CRTIICAL FOR ANNs)
print("Step 2: Standardizing continuous feature matrices....")

Step 2: Standardizing continuous feature matrices....


In [34]:
#Neural networks calculate weights using gradient optimization.
#If input features have highly mismatched scales, the gradients will oscillate, slowing down convergence.
scaler = StandardScaler()
X_train=scaler.fit_transform(X_train_raw)
X_test=scaler.transform(X_test_raw)

ValueError: could not convert string to float: 'C0009'

In [25]:
# 3. TRAINING THE MULTI-LAYER PERCEPTRON (ANN)
print("Step 3: Stacking hidden layers and fitting the network parameters...")


Step 3: Stacking hidden layers and fitting the network parameters...


In [27]:
# hidden_layer_sizes=(16, 8) builds an architecture with 16 neurons in Hidden Layer 1, and 8 neurons in Hidden Layer 2. 'adam' is a robust stochastic gradient optimizer.
mlp_ann = MLPClassifier(
    hidden_layer_sizes=(16, 8),
    activation='relu',
    solver='adam',
    max_iter=500,
    random_state=42
)
mlp_ann.fit(X_train, y_train)

NameError: name 'X_train' is not defined